In [ ]:
!pip install polars
!git config --global user.email "shubhankar.londhe@gmail.com"
!git config --global user.name "ShubhankarLondhe"

In [ ]:
import yaml
import numpy as np
import polars as pl
import polars.selectors as cs
from tqdm import tqdm
import statsmodels.api as sm
from scipy.stats import spearmanr
import functools, operator
from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

## Define analysis params

In [ ]:
mac = 20
variant_class = 'missense'
# ancestries = ['EUR', 'AFR', 'AMR']
ancestries = ['META']

selected_categories = ['missense', 'genetic_diversity', 'conservation', 'gnomad'] # missense

only_snps = True  # Whether to include only SNPs (exclude indels)%%!
only_clinvar = False
exclude_clinvar = False  # Independent toggle: exclude ClinVar-annotated variants

BUCKET_DIR = "/home/jupyter/workspace/aou-gym-processed-data"
DATA_DIR = f"{BUCKET_DIR}/allxall_exome_sumstats"
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'variant_filtering.py').exists())
CONFIG_DIR = str(REPO_ROOT / "configs")

variant_class_path = f"{CONFIG_DIR}/config_variant_classes.yaml"
with open(variant_class_path) as f:
    variant_class_config = yaml.safe_load(f)
    
vc = variant_class_config[variant_class]
vc_filters = vc['variant_filtering']

def compile_filter(node):
    """Turn a YAML filter node into a single pl.Expr."""
    if isinstance(node, str):
        return eval(node)                       # leaf: a Polars expression string
    if isinstance(node, list):
        # bare list defaults to OR (matches the 'coding' semantics)
        return functools.reduce(operator.or_, (compile_filter(n) for n in node))
    if isinstance(node, dict):
        if 'any_of' in node:
            return functools.reduce(operator.or_,  (compile_filter(n) for n in node['any_of']))
        if 'all_of' in node:
            return functools.reduce(operator.and_, (compile_filter(n) for n in node['all_of']))
        raise ValueError(f"filter dict must have 'any_of' or 'all_of', got {list(node)}")
    raise TypeError(f"unexpected filter node: {type(node)}")

print(f"Variant class: {variant_class}")
print(f"  Exclude ClinVar: {exclude_clinvar}")

# Load annotation configuration
config_path = f"{CONFIG_DIR}/config_correlations.yaml" 

with open(config_path) as f:
    config = yaml.safe_load(f)
    
records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = (
    pl.DataFrame(records)
    .filter(
        pl.col("category").is_in(selected_categories)
    )
    .with_columns(
        pl.col("annotation_dir").cast(pl.Int8)
    )
)

all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

## Process gene-trait associations

In [ ]:
RVAT_DIR = f"{BUCKET_DIR}/rvat_gene_results"
ASSOC_FILE = "SAIGE_pLoF_sig_META_MAF0001"
gene_trait_df = pl.read_parquet(f"{RVAT_DIR}/{ASSOC_FILE}_loftee_correlations.parquet")

# FDR needs no nulls in the p-value array
gene_trait_df_quant = (
    gene_trait_df
    .with_columns(
        # region = pl.col('gene_id'),
        # phenotype = pl.col('phenoname'),
        loftee_corr_abs = pl.col('loftee_corr').abs(),
        loftee_corr_dir = pl.col('loftee_corr')/pl.col('loftee_corr').abs(),
    )
    .filter(
        # pl.col("trait_type") == 'continuous'
        pl.col("category").is_in(['physical_measurement', 'lab_measurement'])
    )
    .sort('META_Pvalue_SKATO', descending=False)
    .unique(subset=["region"], keep="first", maintain_order=True)
)

gene_trait_df_dis = (
    gene_trait_df
    .with_columns(
        # region = pl.col('gene_id'),
        # phenotype = pl.col('phenoname'),
        loftee_corr_abs = pl.col('loftee_corr').abs(),
        loftee_corr_dir = pl.col('loftee_corr')/pl.col('loftee_corr').abs(),
    )
    .filter(
        # pl.col("trait_type") != 'continuous'
        pl.col("category").is_in(['mcc2_phecodex','r_drug'])
    )
    .sort('META_Pvalue_SKATO', descending=False)
    .unique(subset=["region"], keep="first", maintain_order=True)
)

gene_trait_df = pl.concat([gene_trait_df_quant, gene_trait_df_dis])
gene_trait_df

## Process annotations

In [ ]:
# anno = pl.scan_parquet("/home/jupyter/workspace/processed_data/ukbgym_exome_sumstats/variant_annotations_exome_all_genes.parquet")
# anno = pl.scan_parquet(f"{DATA_DIR}/variant_metadata_exome_gym_genes_annotated.parquet")
anno = pl.scan_parquet(f"{DATA_DIR}/aou_exome_variant_qc_GYM_annotated_subset_phylop.parquet")

# vc_filters is now the 'variant_filtering' node (list or dict), not a flat list of strings
if vc_filters is not None:
    variant_expr = compile_filter(vc_filters)
else:
    variant_expr = pl.lit(True)

# clinvar / snp filters are still simple ANDs on top
extra = []
if exclude_clinvar:
    extra.append(pl.col('clinical_significance').is_null())
elif only_clinvar:
    extra.append(pl.col('clinical_significance').is_not_null())
if only_snps:
    extra.append((pl.col('ref').str.len_chars()==1) & (pl.col('alt').str.len_chars()==1))
    
anno = (
    anno
    .with_columns(
        is_ins = pl.col('ref').str.len_chars() < pl.col('alt').str.len_chars(),
        is_del = pl.col('ref').str.len_chars() > pl.col('alt').str.len_chars(),
    )
    .filter(
        # Always-applied filters
        (pl.col('region').is_in(gene_trait_df['region'].unique())),
        (pl.col('variant_length') <= 50),

        # Dynamic filters from variant_class.yaml + exclude_clinvar
        variant_expr,        # the OR/AND tree, as one expression
        *extra,              # these AND with everything
    )
    .with_columns(
        inframe_deletion = (pl.col('variant_length') % 3 == 1) & (pl.col('is_del')==True), #SNPs have length 1
        inframe_insertion = (pl.col('variant_length') % 3 == 1) & (pl.col('is_ins')==True),

        clinvar_patho = pl.col('clinical_significance').str.contains('(?i)Pathogenic').fill_null(False),
        clinvar_likely_patho = pl.col('clinical_significance').str.contains('(?i)Likely_pathogenic').fill_null(False),
        clinvar_benign = pl.col('clinical_significance').str.contains('(?i)Benign').fill_null(False),
        clinvar_likely_benign = pl.col('clinical_significance').str.contains('(?i)Likely_benign').fill_null(False),
    )
)

selected_annos = anno_config_df.filter(
    pl.col('category').is_in(selected_categories)
)['annotation'].to_list()

existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]
selected_annos = list(set(selected_annos).intersection(set(existing_annos)))
fillna_cols = [c+'_is_na' for c in selected_annos]

anno = (
    anno
    .select(
        set(['id', 'region']).union(set(selected_annos))
    )
    .collect(engine='streaming')
    # .drop_nulls()
)

anno

In [ ]:
melted_anno = (
    anno.lazy()

    .select(
        set(['id', 'region']).union(set(selected_annos))
    )

    .unpivot(
        index=["id", "region"],
        on=selected_annos,
        variable_name="annotation",
        value_name="annotation_score"
    )
    .with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )
        
    # Keep variants that don't have fillna annotation
    # .join(
    #     anno_fillna_melted,
    #     on=['id', 'region', 'annotation'],
    #     how='semi'
    # )
    .drop_nulls()
    
    .collect(engine='streaming')
)

melted_anno

## Process phenotypes (appv file)

In [ ]:
appv_list = [
    'appv_physical_measurement.parquet',
    'appv_lab_measurement.parquet',
    # 'appv_mcc2_phecodex.parquet',
    # 'appv_r_drug.parquet'
]

pheno_appv = (
    pl.concat(
        [pl.scan_parquet(f"{DATA_DIR}/variant_sumstats_associations_META/{appv_file}") for appv_file in appv_list]
    )
    .select(['id', 'phenotype', 'AC', 'BETA', 'SE'])
)


# Filter to variants in annotation set and low MAC
anno_keys = melted_anno.select(pl.col('id').unique()).lazy()

# Merge phenotype data and annotation data
appv = (
    pheno_appv
    .join(
        anno_keys, on='id', how='semi'
    )
    .filter(
        pl.col('AC') <= mac,
    )
)

appv.head().collect()

## Join and get stats

In [ ]:
id_region = anno.select(['id', 'region']).unique().lazy()

# Build the main lazy query plan
# Join order: appv → id_region (adds region) → gene_trait_df (filter early) → melted_anno (annotation scores)
final_lazy_plan = (
    appv
    .join(id_region, on='id', how='inner')
    .join(
        gene_trait_df[["region", "phenotype"]].lazy(),
        on=["region", "phenotype"],
        # gene_trait_df.select(["ancestry", "region", "phenotype"]).lazy(),
        # on=["ancestry", "region", "phenotype"],
        how="inner"
    )
    .join(
        melted_anno.lazy(),
        on=["id", "region"],
        how="inner"
    )

    # Spearman: rank with "average" for proper tie handling
    .with_columns(
        pl.col(c)
        .rank("average")
        .over(["region", "phenotype", "annotation"])
        # .over(["ancestry", "region", "phenotype", "annotation"])
        .alias(f"{c}_rank")
        for c in ['annotation_score', 'BETA']
    )

    .group_by(["region", "phenotype", "annotation"])
    # .group_by(["ancestry", "annotation", "region", "phenotype"])
    .agg(
        n_variants = pl.col("id").count(),
        correlation = pl.when(
            (pl.col("annotation_score_rank").n_unique() > 1) & 
            (pl.col("BETA_rank").n_unique() > 1)
        )
        .then(
            pl.corr("annotation_score_rank", "BETA_rank", propagate_nans=True)
        )
        .otherwise(None)
    )

    # .select(["ancestry", "region", "phenotype", "annotation", "n_variants", "correlation"])
    
    .drop_nans().drop_nulls()
        
    .collect(engine='streaming')
)

# Join with beta directions and annotation directions
correlation_df = (
    final_lazy_plan
    
    .join(
        anno_config_df.filter(pl.col("category").is_in(selected_categories)),
        on='annotation'
    )
    .join(
        # gene_trait_df.select(["ancestry", "region", "phenotype", "loftee_corr", "loftee_corr_dir"]),
        # on=['ancestry', 'region', 'phenotype'],
        gene_trait_df.select(["region", "phenotype", "loftee_corr_dir"]),
        on=['region', 'phenotype'],
    )
    .with_columns(
        corr_beta = pl.col('correlation')*pl.col('loftee_corr_dir')*pl.col('annotation_dir') 
    )
)

print("Final DataFrame shape:", correlation_df.shape)
correlation_df

In [ ]:
consistent_gt = (
    correlation_df
    .drop_nulls().drop_nans()
    # .group_by(['ancestry', 'region', 'phenotype'])
    .group_by(['region', 'phenotype'])
    .agg(n_annotations = pl.len())
    # .sort('n_annotations')
    .filter(pl.col('n_annotations') == correlation_df['annotation'].n_unique())
)

print(f"{consistent_gt.shape[0]} pairs of ancestry-gene-phenotype have correlations for all annotations")

filt_corr_df = correlation_df.filter(pl.col("n_variants") > 10)
# filt_corr_df = filt_corr_df.join(consistent_gt, on=['ancestry', 'region', 'phenotype'], how='inner')
filt_corr_df = filt_corr_df.join(consistent_gt, on=['region', 'phenotype'], how='inner')
filt_corr_df

## Plot

In [ ]:
plotting_col = 'corr_beta'
dashed_line_value = 1 if 'rescaled' in plotting_col else 0

if exclude_clinvar:
    plot_title = f"{vc['x_label']} (no ClinVar)"
elif only_clinvar:
    plot_title = f"{vc['x_label']} (only ClinVar)"
else:
    plot_title = vc['x_label']

cat_colors = pl.DataFrame({
    "category": config["correlation_categories"].keys(),
    "color": config["correlation_categories"].values(),
})
filt_corr_df = (
    filt_corr_df
    .drop('color')
    .join(
        cat_colors,
        on='category',
        how='left'
    )
)

# Calculate medians and join back
exclude_annos = []
plot_corr_pl = (
    filt_corr_df
    .drop_nans()
    .filter(~pl.col('annotation').is_in(exclude_annos))
    .with_columns(
        median_corr_beta = pl.col(plotting_col).median().over("annotation")
    )
)

# 2. Determine the categorical order for the labels
ordered_labels = (
    plot_corr_pl
    .sort("median_corr_beta", descending=False)
    .select("label")
    .unique(maintain_order=True)
    .to_series()
)

# 3. Apply the ordering using pl.Enum
plot_corr_pl = plot_corr_pl.with_columns(pl.col("label").cast(pl.Enum(ordered_labels)))

# 4. Create the color dictionary (Polars style)
color_dict = dict(plot_corr_pl.select("annotation", "color").unique().iter_rows())

plot_corr_pl

In [ ]:
# Plot
(
    ggplot(plot_corr_pl, aes(x="label", y=plotting_col, fill="annotation"))
    + geom_hline(aes(yintercept=dashed_line_value), color='black', linetype='dotted')
    + geom_boxplot(alpha=1, outlier_shape=None)
    + theme_minimal()
    + scale_fill_manual(values=color_dict)
    + labs(
        x="",
        y="Spearman correlation",
        title=f"{plot_title} variants - ({plot_corr_pl['region'].n_unique()} assocs.)",
    )
    + coord_flip()
    + theme(
        figure_size=(8, plot_corr_pl['annotation'].n_unique()/3 + 0.5),
        legend_position="none",
        axis_text=element_text(size=11),
        axis_title=element_text(size=11),
        legend_text=element_text(size=11),
        legend_title=element_text(size=11),
        panel_grid_major=element_line(color='lightgray', size=0.5),
        panel_grid_minor=element_line(color='lightgray', size=0.25),
        plot_background=element_rect(fill="white", color="white"),
    )
)

In [ ]:
import pandas as pd
import itertools
import numpy as np
from scipy import stats

# --- 1. Annotation set + label map ---
plot_annotations = plot_corr_pl.select("annotation").unique().to_series().to_list()
plot_annotations = [a for a in plot_annotations if a not in exclude_annos]
anno_to_label = dict(plot_corr_pl.select(["annotation", "label"]).unique().iter_rows())

# --- 2. Precompute every head-to-head paired comparison once ---
# diff[(a, b)] = mean(corr_beta_a - corr_beta_b) over genes/traits shared by a and b
# pval[{a, b}] = Wilcoxon signed-rank p-value (symmetric)
vals_by_ann = {
    a: plot_corr_pl.filter(pl.col("annotation") == a).select(["region", "phenotype", "corr_beta"])
    for a in plot_annotations
}
ALPHA = 0.05
diff, pval = {}, {}
for a, b in itertools.combinations(plot_annotations, 2):
    paired = vals_by_ann[a].join(vals_by_ann[b], on=["region", "phenotype"], suffix="_b")
    ca = paired["corr_beta"].to_numpy()
    cb = paired["corr_beta_b"].to_numpy()
    d = float(ca.mean() - cb.mean()) if len(ca) else 0.0
    if len(ca) >= 10 and not np.allclose(ca, cb):
        _, pv = stats.wilcoxon(ca, cb, alternative="two-sided")
    else:
        pv = 1.0
    diff[(a, b)], diff[(b, a)] = d, -d
    pval[frozenset((a, b))] = pv

def beats(a, b):  # a is SIGNIFICANTLY better than b (higher corr on shared genes)
    return pval[frozenset((a, b))] < ALPHA and diff[(a, b)] > 0

# --- 3. Dominance ordering (upset-safe topological sort) ---
wins      = {t: sum(beats(t, u) for u in plot_annotations if u != t) for t in plot_annotations}
losses    = {t: sum(beats(u, t) for u in plot_annotations if u != t) for t in plot_annotations}
advantage = {t: sum(diff[(t, u)] for u in plot_annotations if u != t) for t in plot_annotations}

# A tool may be placed (ranked next-highest) only once every tool that
# significantly beats it is already placed -> no lower tool ever beats a
# higher one. Among placeable tools (on par with all still-unplaced ones),
# prefer fewest losses, then most wins, then largest margin.
def _key(t):
    return (losses[t], -wins[t], -advantage[t])

remaining, ordered_tools = set(plot_annotations), []
while remaining:
    placeable = [t for t in remaining
                 if not any(beats(u, t) for u in remaining if u != t)]
    if not placeable:                       # only triggers on a true cycle
        placeable = list(remaining)
    nxt = min(placeable, key=_key)
    ordered_tools.append(nxt)
    remaining.discard(nxt)

ordered_labels = [anno_to_label.get(t, t) for t in ordered_tools]

# Should be empty unless a cycle was force-broken above.
upsets = [(ordered_tools[j], ordered_tools[i])
          for i in range(len(ordered_tools)) for j in range(i + 1, len(ordered_tools))
          if beats(ordered_tools[j], ordered_tools[i])]

print("Tool order (best -> worst):")
for t in ordered_tools:
    print(f"  {anno_to_label.get(t, t):<22} losses={losses[t]}  wins={wins[t]}  margin={advantage[t]:+.3f}")
print(f"\nSignificant upsets (lower tool beats a higher one): {len(upsets)}")
for lo, hi in upsets:
    print(f"  {anno_to_label.get(lo, lo)} > {anno_to_label.get(hi, hi)} (p={pval[frozenset((lo, hi))]:.2g})")

# --- 4. Build heatmap from the precomputed comparisons (no recompute) ---
heatmap_data = []
for ann_x, ann_y in itertools.product(ordered_tools, repeat=2):
    label_x = anno_to_label.get(ann_x, ann_x)
    label_y = anno_to_label.get(ann_y, ann_y)
    if ann_x == ann_y:
        heatmap_data.append({'Tool_X': label_x, 'Tool_Y': label_y, 'mean_diff': 0.0, 'sig': ''})
        continue
    md = diff[(ann_y, ann_x)]                  # mean(Tool_Y) - mean(Tool_X), matches original fill
    pv = pval[frozenset((ann_x, ann_y))]
    sig = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else ''
    heatmap_data.append({'Tool_X': label_x, 'Tool_Y': label_y, 'mean_diff': md, 'sig': sig})

df_heat = pd.DataFrame(heatmap_data)

# Lock in categorical order so best tools appear top/right
df_heat['Tool_X'] = pd.Categorical(df_heat['Tool_X'], categories=ordered_labels,       ordered=True)
df_heat['Tool_Y'] = pd.Categorical(df_heat['Tool_Y'], categories=ordered_labels[::-1], ordered=True)

n_tools = len(ordered_tools)

print(appv_list)
# --- 5. Plot ---
plot = (
    ggplot(df_heat, aes(x='Tool_X', y='Tool_Y', fill='mean_diff'))
    + geom_tile(color="#FFFFFF", size=0.5)
    + geom_text(aes(label='sig'), color="black", size=12, va='center', nudge_y=-0.1)
    + scale_fill_gradient2(low="#2C7BB6", mid="#FFFFFF", high="#D7191C", midpoint=0)
    + labs(
        title=f"{plot_title} variants - ({plot_corr_pl['region'].n_unique()} assocs.)",
        subtitle="Wilcoxon signed-rank test on paired per-gene-trait corr_beta\n*** p<0.001, ** p<0.01, * p<0.05",
        x="Tool X",
        y="Tool Y",
        fill="Tool Y − X\n(avg Spearman corr)"
    )
    + theme_minimal()
    + theme(
        figure_size=(n_tools * 0.6 + 2.5, n_tools * 0.6 + 1.5),
        aspect_ratio=1,
        axis_title=element_text(size=13),
        axis_text=element_text(size=13),
        axis_text_x=element_text(rotation=45, hjust=1),
        panel_grid=element_blank(),
        legend_background=element_rect(fill="white", color='white', alpha=0.8),
        plot_background=element_rect(fill="white", color="white"),
    )
)

FIG_DIR = str(REPO_ROOT / "paper_figures")
plot.save(f"{FIG_DIR}/F3_pheno_corr_allxall.svg", dpi=200)

plot

In [ ]:
not_in_paper_annos = ['Mammalian PhyloP', 'Primate PhyloP', 'popEVE', 'SIFT', 'gnomADe AF', 'gnomADg AF']

df_heat_small = df_heat[
    ~df_heat['Tool_X'].isin(not_in_paper_annos) & 
    ~df_heat['Tool_Y'].isin(not_in_paper_annos)
]

n_tools = len(df_heat_small['Tool_X'].unique())

print(appv_list)
plot_small = (
    ggplot(
        df_heat_small, 
        aes(x='Tool_X', y='Tool_Y', fill='mean_diff'))
    + geom_tile(color="#FFFFFF", size=0.5)
    + geom_text(aes(label='sig'), color="black", size=12, va='center', nudge_y=-0.1)
    + scale_fill_gradient2(low="#2C7BB6", mid="#FFFFFF", high="#D7191C", midpoint=0)
    + labs(
        title=f"{plot_title} variants - ({plot_corr_pl['region'].n_unique()} assocs.)",
        x="Tool X",
        y="Tool Y",
        fill="Tool Y − X\n(avg Spearman corr)"
    )
    + theme_minimal()
    + theme(
        figure_size=(n_tools * 0.6 + 2.5, n_tools * 0.6 + 1.5),
        aspect_ratio=1,
        axis_title=element_text(size=13),
        axis_text=element_text(size=13),
        axis_text_x=element_text(rotation=45, hjust=1),
        panel_grid=element_blank(),
        legend_background=element_rect(fill="white", color='white', alpha=0.8),
        plot_background=element_rect(fill="white", color="white"),
    )
)

plot_small

## Consistency with Full data

In [ ]:
from dataframe_from_heatmap import get_heat_df

df_full_data = get_heat_df().to_pandas()
df_full_data['Tool_X'] = pd.Categorical(df_full_data['Tool_X'], categories=ordered_labels, ordered=True)
df_full_data['Tool_X'] = pd.Categorical(df_full_data['Tool_X'], categories=ordered_labels, ordered=True)

df_full_data

In [ ]:
agree_df = df_full_data.merge(
    df_heat,
    on=["Tool_X", "Tool_Y"],
    how="inner",
)

agree_df

In [ ]:
agree_plot = (
    pl.from_pandas(agree_df)
    .with_columns(
        both_significant = (pl.col("full_data_sig") != "") & (pl.col("sig") != ""),
        genebass_dir = pl.col("mean_diff")/pl.col("mean_diff").abs()
    )
    .with_columns(
        agree_category = pl.when(
                pl.col("both_significant") & (pl.col("genebass_dir") == pl.col("full_data_dir"))
            ).then(pl.lit("concordant"))
            .otherwise(
                pl.when(
                    pl.col("both_significant") & (pl.col("genebass_dir") != pl.col("full_data_dir"))
                ).then(pl.lit("discordant"))
                .otherwise(pl.lit("neither"))
            )
    )
    .to_pandas()
)
agree_plot

In [ ]:
TOOLS = ["CPT-1", "BayesDel", "REVEL", "ClinPred", "AlphaMissense", "CADD Raw", "PolyPhen2", "ESM1v", "GPN-MSA", "Vertebrate PhyloP"]


# fixed axis order (ranking) + legend order
agree_df["Tool_X"] = pd.Categorical(agree_df["Tool_X"], categories=TOOLS, ordered=True)
agree_df["Tool_Y"] = pd.Categorical(agree_df["Tool_Y"], categories=TOOLS, ordered=True)

agree_plot["category"] = pd.Categorical(
    agree_plot["agree_category"],
    categories=["concordant", "significant reversal", "neither"], 
    ordered=True)

cat_colors = {
    "concordant":          "#8FBF9F",  # green
    "significant reversal": "#B02A1E",  # red
    "neither":             "#FFFFFF",  # white
}

n_tools = agree_plot["Tool_X"].nunique()

TOOLS = ["CPT-1", "BayesDel", "REVEL", "ClinPred", "AlphaMissense",
         "CADD Raw", "PolyPhen2", "ESM1v", "GPN-MSA", "Vertebrate PhyloP"]

plot = (
    ggplot(agree_plot, aes(x="Tool_X", y="Tool_Y", fill="category"))
    + geom_tile(color="#EEEEEE", size=0.25)
    + scale_fill_manual(values=cat_colors, name="Category")
    + scale_x_discrete(limits=TOOLS)          # CPT-1 left  -> Vertebrate PhyloP right
    + scale_y_discrete(limits=TOOLS[::-1])    # CPT-1 top   -> Vertebrate PhyloP bottom
    + labs(title="Full data vs All x All", x="", y="")
    + theme_minimal()
    + theme(
        figure_size=(len(TOOLS) * 0.6 + 2.5, len(TOOLS) * 0.6 + 1.5),
        aspect_ratio=1,
        axis_title=element_text(size=13),
        axis_text=element_text(size=13),
        axis_text_x=element_text(rotation=45, hjust=1),
        panel_grid=element_blank(),
        legend_background=element_rect(fill="white", color="white", alpha=0.8),
        plot_background=element_rect(fill="white", color="white"),
    )
)

FIG_DIR = str(REPO_ROOT / "paper_figures")
plot.save(f"{FIG_DIR}/F3_correlation_concordance_allxall.svg", dpi=200)

plot